# wandb-finish — worked example 3: Run multiple sweep trials with init/finish pairs

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `wandb-finish`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

In a wandb sweep, the sweep agent calls your training function multiple times — once per trial. Each call must open a new run with `wandb.init()` and close it with `wandb.finish()` before the next trial starts. Without `finish()`, the second `wandb.init()` sees a stale active run and may produce a warning or create a nested run.

## Worked solution

**Step 1 — outer loop over trials.**
We iterate over a list of hyperparameter dicts, one per trial. Each dict represents what the sweep agent would have sampled.

**Step 2 — init inside each trial.**
At the start of each iteration, we call `wandb.init(project=..., config=trial_cfg)`. This opens a fresh run for this trial's configuration.

**Step 3 — training, then finish.**
After the fake training, we call `wandb.finish()`. This closes the current run. The next iteration's `wandb.init()` will then find no active run and start cleanly.

**Step 4 — verify call count.**
After `N` trials, both `wandb.init` and `wandb.finish` should have been called exactly `N` times.

In [ ]:
import sys
from unittest.mock import MagicMock
sys.modules.setdefault('wandb', MagicMock())
import wandb

def run_sweep_trials(trials):
    """
    trials: list of dicts, each dict is a hyperparameter config for one trial.
    Returns: number of trials completed.
    """
    for i, trial_cfg in enumerate(trials):
        # Open a new run for this trial
        wandb.init(project='sweep-demo', config=trial_cfg, name=f'trial-{i}')
        
        # Fake training
        fake_loss = trial_cfg.get('lr', 1e-3) * 100
        
        # Close the run before the next trial
        wandb.finish()
    
    return len(trials)

# Exercise it
trial_configs = [
    {'lr': 1e-3, 'batch_size': 32},
    {'lr': 5e-4, 'batch_size': 64},
    {'lr': 1e-4, 'batch_size': 128},
]
wandb.init.reset_mock()
wandb.finish.reset_mock()

n_completed = run_sweep_trials(trial_configs)
print('Trials completed:', n_completed)
print('wandb.init call count:', wandb.init.call_count)    # should be 3
print('wandb.finish call count:', wandb.finish.call_count)  # should be 3
print('Paired correctly:', wandb.init.call_count == wandb.finish.call_count)